<a href="https://colab.research.google.com/github/avinseth/Prompt-Engineering-Techniques/blob/Coding-Hub/Gemini_Guided_practice_building_llm_apps_day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-google-genai
!pip install langchain-text-splitters
!pip install google-generativeai
!pip install faiss-cpu
!pip install sentence-transformers
!pip install pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.1
    Uninstalling langchain-core-1.2.1:
      Successfully uninstalled langchain-core-1.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is inc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 6.1 MB/s eta 0:00:00


In [ ]:
# -*- coding: utf-8 -*-
"""
Guided_Practice_Loader_Splitter_Embeddings_Gemini.py

=====================================================
LangChain Loader, Splitter, and Embeddings (Gemini)
=====================================================
"""

import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
import faiss
import warnings
warnings.filterwarnings("ignore")


# =====================================================
# Step 0: Set Gemini API Key
# =====================================================

os.environ["GOOGLE_API_KEY"] = ""


# =====================================================
# Step 1: Load text data using TextLoader
# =====================================================

text_loader = TextLoader("state_of_union.txt")
text_documents = text_loader.load()

print(text_documents[0].page_content[:100])  # First 100 characters







Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and th


In [ ]:
# =====================================================
# Step 2: Load PDF using PyPDFLoader
# =====================================================

pdf_loader = PyPDFLoader("michael_resume.pdf")
pdf_pages = pdf_loader.load_and_split()

print(pdf_pages[0].page_content[:100])  # First 100 characters of first page




CURRICULUM VITAE :  
M ichael M . Scott OBE, B.Sc., Dip.Ed  
 
Home address:  Strome House     Date 


In [ ]:
# =====================================================
# Step 3: Split documents into chunks
# =====================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,
    chunk_overlap=64
)

split_docs = text_splitter.split_documents(pdf_pages)

print("Number of chunks:", len(split_docs))




Number of chunks: 15


In [ ]:
# =====================================================
# Step 4: HuggingFace Embeddings
# =====================================================

MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

hf_embeddings = HuggingFaceEmbeddings(model_name=MODEL_NAME)

sample_text = split_docs[0].page_content
hf_embedding_result = hf_embeddings.embed_documents([sample_text])

print("HF embedding length:", len(hf_embedding_result[0]))




modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HF embedding length: 768


In [ ]:
# =====================================================
# Step 6: Create FAISS Vector Store
# =====================================================

faiss_index = FAISS.from_documents(
    split_docs,
    hf_embeddings
)





In [ ]:
# =====================================================
# Step 7: Similarity Search
# =====================================================

query = "What is the candidate's skill sets?"

results = faiss_index.similarity_search_with_score(
    query,
    k=2
)

print("\nTop 2 Similar Documents:")
for doc, score in results:
    print("Score:", score)
    print(doc.page_content[:300])
    print("-" * 50)